In [ ]:
# obtain gguf
# read set values from yaml
!set model_url
!set GIT_LFS_SKIP_SMUDGE=1 && git clone $model_url
!set model_name=regex model_url_rfind_slash
!python llama.cpp/convert_hf_to_gguf.py $model_name --outfile ./$model_name+'.gguf'
!llama-quantize $model_name.gguf $model_name+$quantization_type.gguf $quantization_type N=?-1?

In [ ]:
import benchmark
import time
for model_name in ["olmo2_IQ4_XS"]:#,"smolm2-pruned-sparsegpt-q4km"]:
    model_path=f'/mnt/d/Inno/Thesis/out/olmo2/olmo2_IQ4_XS.gguf'
    benchllm=benchmark.BenchLLM('../../llama.cpp/build/bin/llama-server','../../llama.cpp/build/bin/llama-perplexity',model_path)
    time.sleep(10) # wait for previous model free and server to close
    benchllm.start_server()
    time.sleep(150) # wait for model to load
    benchllm.bench() # stop server itself
    benchllm.save_to_file(f'./{model_name}')

In [ ]:
model_path=f'/mnt/d/Inno/Thesis/{model_name}.gguf'
benchllm=benchmark.BenchLLM('../../llama.cpp/bin/llama-server','../../llama.cpp/bin/llama-perplexity',model_path)
time.sleep(10) # wait for previous model free and server to close
benchllm.start_server()

In [1]:
from pathlib import Path 
folder_path=Path("../../out/olmo2")
[str(path.parent) for path in folder_path.rglob("quantize_config.json")] + [str(path) for path in folder_path.rglob("*.gguf")]

['../../out/olmo2/quantized/olmo2_unstructured_wanda_0.2_gptq_4',
 '../../out/olmo2/quantized/olmo2_unstructured_wanda_0.2_gptq_8',
 '../../out/olmo2/quantized/olmo2_gptq_8',
 '../../out/olmo2/quantized/olmo2_unstructured_wanda_0.5_gptq_8',
 '../../out/olmo2/quantized/olmo2_unstructured_sparsegpt_0.5_gptq_8',
 '../../out/olmo2/quantized/olmo2_unstructured_sparsegpt_0.2_gptq_4',
 '../../out/olmo2/quantized/olmo2_unstructured_wanda_0.5_gptq_4',
 '../../out/olmo2/quantized/olmo2_gptq_4',
 '../../out/olmo2/quantized/olmo2_unstructured_sparsegpt_0.5_gptq_4',
 '../../out/olmo2/quantized/olmo2_unstructured_sparsegpt_0.2_gptq_8',
 '../../out/olmo2/original/olmo2_q16.gguf',
 '../../out/olmo2/pruned/unstructured/wanda/0.2/olmo2_unstructured_wanda_0.2_q16.gguf',
 '../../out/olmo2/pruned/unstructured/wanda/0.5/olmo2_unstructured_wanda_0.5_q16.gguf',
 '../../out/olmo2/pruned/unstructured/sparsegpt/0.2/olmo2_unstructured_sparsegpt_0.2_q16.gguf',
 '../../out/olmo2/pruned/unstructured/sparsegpt/0.5/ol

In [3]:
import benchmark_client
benchllm=benchmark_client.BenchLLMClient('##','##')
result_df=benchllm.compare_results('../../out/smollm2/benchmark_results/out/smollm2','../../out/smollm2/benchmark_results/out/smollm2/original/smollm2_q16.gguf.json')
# p value is calculated for accuracy with McNemar's test p-value relative to base model
result_df

2026-04-29 16:45:26,565 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/hotpotqa/hotpot_qa/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-29 16:45:26,615 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/hotpotqa/hotpot_qa/1908d6afbbead072334abe2965f91bd2709910ab/README.md "HTTP/1.1 200 OK"
2026-04-29 16:45:26,790 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/hotpotqa/hotpot_qa/resolve/1908d6afbbead072334abe2965f91bd2709910ab/hotpot_qa.py "HTTP/1.1 404 Not Found"
2026-04-29 16:45:27,230 - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/hotpotqa/hotpot_qa/hotpotqa/hotpot_qa.py "HTTP/1.1 404 Not Found"
2026-04-29 16:45:27,349 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/hotpotqa/hotpot_qa/resolve/1908d6afbbead072334abe2965f91bd2709910ab/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-04-29 16:45:27,618 - INFO - HTTP Request: GET https://datasets-server.huggi

!


,method,accuracy,ppl,prefill,decode,cor->incor,incor->cor,p_value (mcNemar),CI (95%)
0,smollm2_q16.gguf.json,0.30,9.68+-0.50,37.66+-2.20,9.29+-0.68,0.00,0.00,"nan,n=200","nan,nan"
1,smollm2_unstructured_sparsegpt_0.5_gptq_4.json,0.23,38.92+-27.06,46.29+-6.39,24.95+-5.81,0.54,0.24,"0.02,n=200","0.01,0.13"
2,smollm2_gptq_8.json,0.31,22.16+-17.15,56.01+-10.17,19.94+-4.09,0.05,0.08,"0.48,n=200","-0.04,0.02"
3,smollm2_unstructured_wanda_0.2_gptq_4.json,0.32,24.11+-16.05,34.23+-4.05,34.50+-10.82,0.20,0.27,"0.47,n=200","-0.07,0.03"
4,smollm2_unstructured_wanda_0.5_gptq_8.json,0.21,32.94+-21.85,53.19+-9.70,18.50+-4.38,0.58,0.19,"0.00,n=200","0.03,0.14"
5,smollm2_unstructured_wanda_0.5_gptq_4.json,0.26,38.85+-25.40,43.41+-6.01,27.53+-40.47,0.37,0.21,"0.14,n=200","-0.01,0.09"
6,smollm2_gptq_4.json,0.29,23.98+-15.82,44.22+-6.41,29.27+-6.13,0.21,0.17,"0.67,n=200","-0.04,0.06"
7,smollm2_unstructured_sparsegpt_0.5_gptq_8.json,0.21,32.63+-21.32,53.32+-10.01,18.53+-4.29,0.58,0.19,"0.00,n=200","0.03,0.14"
8,smollm2_unstructured_sparsegpt_0.2_gptq_8.json,0.32,21.57+-14.60,50.56+-10.58,18.59+-4.59,0.13,0.17,"0.49,n=200","-0.06,0.03"
9,smollm2_unstructured_sparsegpt_0.2_gptq_4.json,0.30,24.51+-16.43,34.23+-4.05,20.47+-5.56,0.21,0.23,"0.85,n=200","-0.06,0.05"


In [ ]:
benchllm.calc_corr(result_df)

In [2]:
import benchmark_client
benchllm=benchmark_client.BenchLLMClient('##','##')
result_df=benchllm.compare_results('../../out/olmo2/benchmark_results/out/olmo2','../../out/olmo2/benchmark_results/out/olmo2/original/olmo2_q16.gguf.json')
result_df

2026-04-29 16:44:56,713 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/hotpotqa/hotpot_qa/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-04-29 16:44:56,756 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/hotpotqa/hotpot_qa/1908d6afbbead072334abe2965f91bd2709910ab/README.md "HTTP/1.1 200 OK"
2026-04-29 16:44:56,927 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/hotpotqa/hotpot_qa/resolve/1908d6afbbead072334abe2965f91bd2709910ab/hotpot_qa.py "HTTP/1.1 404 Not Found"
2026-04-29 16:44:57,402 - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/hotpotqa/hotpot_qa/hotpotqa/hotpot_qa.py "HTTP/1.1 404 Not Found"
2026-04-29 16:44:57,520 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/hotpotqa/hotpot_qa/resolve/1908d6afbbead072334abe2965f91bd2709910ab/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-04-29 16:44:57,795 - INFO - HTTP Request: GET https://datasets-server.huggi

!


,method,accuracy,ppl,prefill,decode,cor->incor,incor->cor,p_value (mcNemar),CI (95%)
0,olmo2_q16.gguf.json,0.14,13.81+-0.86,58.64+-4.50,12.78+-1.15,0.00,0.00,"nan,n=200","nan,nan"
1,olmo2_unstructured_sparsegpt_0.5_gptq_4.json,0.27,38.57+-23.48,60.89+-6.28,45.24+-10.71,0.22,0.69,"0.00,n=200","-0.19,-0.06"
2,olmo2_unstructured_wanda_0.5_gptq_8.json,0.24,37.66+-21.73,60.85+-10.33,23.48+-8.18,0.35,0.75,"0.01,n=200","-0.17,-0.02"
3,olmo2_unstructured_wanda_0.2_gptq_8.json,0.29,30.68+-17.56,61.85+-9.51,31.27+-7.87,0.21,0.71,"0.00,n=200","-0.22,-0.07"
4,olmo2_gptq_4.json,0.28,24.16+-16.74,63.12+-8.95,25.85+-3.80,0.23,0.72,"0.00,n=200","-0.21,-0.07"
5,olmo2_unstructured_sparsegpt_0.2_gptq_4.json,0.28,33.22+-18.81,70.82+-6.42,30.54+-11.40,0.18,0.66,"0.00,n=200","-0.20,-0.07"
6,olmo2_unstructured_wanda_0.2_gptq_4.json,0.32,24.11+-16.05,53.27+-7.28,59.14+-12.93,0.17,0.72,"0.00,n=200","-0.25,-0.10"
7,olmo2_unstructured_sparsegpt_0.5_gptq_8.json,0.24,37.07+-23.09,60.85+-10.33,23.41+-8.14,0.31,0.71,"0.01,n=200","-0.16,-0.03"
8,olmo2_unstructured_sparsegpt_0.2_gptq_8.json,0.28,31.31+-17.89,68.35+-13.53,29.15+-5.54,0.21,0.70,"0.00,n=200","-0.21,-0.07"
9,olmo2_gptq_8.json,0.14,21.73+-16.54,71.48+-16.22,30.39+-6.31,0.69,0.69,"1.00,n=200","-0.06,0.06"


In [7]:
benchllm.calc_corr(result_df)

,accuracy,ppl,prefill,decode,fail
accuracy,1.00000000000000000000,-0.16330923481015682808,0.03603748475667356566,-0.00596154400760356019,-0.40609140151650124917
ppl,-0.16330923481015682808,1.00000000000000000000,-0.05313132587145032865,0.19299858317697721199,0.31755720063898279504
prefill,0.03603748475667356566,-0.05313132587145032865,1.00000000000000000000,-0.76872987865107467176,0.07577854327597413620
decode,-0.00596154400760356019,0.19299858317697721199,-0.76872987865107467176,1.00000000000000000000,0.36737054663493851070
fail,-0.40609140151650124917,0.31755720063898279504,0.07577854327597413620,0.36737054663493851070,1.00000000000000000000
